In [1]:
import sys
print(sys.executable)

/Users/dipasmitadebroy/Documents/Artificial_Intelligence/ml_projects/movie-recommender/venv/bin/python


In [2]:
# Import for the Neural CF Challenger model

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import mean_squared_error, mean_absolute_error
from tqdm.notebook import tqdm
import joblib, warnings, os

warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device: {device}")

✅ Device: cpu


In [3]:
# Config and load

PROCESSED_DIR = "../data/processed/"
MODEL_DIR     = "../models/"
FIGURES_DIR   = "../reports/figures/"

EMBED_DIM  = 64
HIDDEN     = [128, 64, 32]
DROPOUT    = 0.2
LR         = 1e-3
BATCH_SIZE = 512
EPOCHS     = 10

train = pd.read_parquet(PROCESSED_DIR + "train.parquet")
test  = pd.read_parquet(PROCESSED_DIR + "test.parquet")
print(f"Train: {len(train):,} | Test: {len(test):,}")

Train: 19,784,257 | Test: 5,026,226


In [4]:
# Encode IDs

user2idx  = {u: i for i, u in enumerate(train["userId"].unique())}
movie2idx = {m: i for i, m in enumerate(train["movieId"].unique())}

train["user_idx"]  = train["userId"].map(user2idx)
train["movie_idx"] = train["movieId"].map(movie2idx)

test["user_idx"]  = test["userId"].map(user2idx)
test["movie_idx"] = test["movieId"].map(movie2idx)

test_enc = test.dropna(subset=["user_idx","movie_idx"]).copy()
test_enc[["user_idx","movie_idx"]] = test_enc[["user_idx","movie_idx"]].astype(int)

N_USERS, N_MOVIES = len(user2idx), len(movie2idx)
print(f"N users: {N_USERS:,} | N movies: {N_MOVIES:,}")
print(f"Test rows after encode filter: {len(test_enc):,}")

N users: 162,541 | N movies: 18,429
Test rows after encode filter: 5,026,171


In [5]:
# Pytorch dataset and dataloader

class RatingsDataset(Dataset):
    def __init__(self, df):
        self.users   = torch.LongTensor(df["user_idx"].values)
        self.movies  = torch.LongTensor(df["movie_idx"].values)
        self.ratings = torch.FloatTensor(df["rating"].values)
    def __len__(self): return len(self.ratings)
    def __getitem__(self, i): return self.users[i], self.movies[i], self.ratings[i]

train_dl = DataLoader(RatingsDataset(train), batch_size=BATCH_SIZE, shuffle=True,  num_workers=0)
test_dl  = DataLoader(RatingsDataset(test_enc), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f"Train batches: {len(train_dl)} | Test batches: {len(test_dl)}")

Train batches: 38642 | Test batches: 9817


In [6]:
# Model Architecture (GMF + MLP)

class NCFNet(nn.Module):
    def __init__(self, n_users, n_items, embed_dim, hidden, dropout):
        super().__init__()
        # GMF branch
        self.gmf_u = nn.Embedding(n_users, embed_dim)
        self.gmf_i = nn.Embedding(n_items, embed_dim)
        # MLP branch
        self.mlp_u = nn.Embedding(n_users, embed_dim)
        self.mlp_i = nn.Embedding(n_items, embed_dim)
        layers, d = [], embed_dim * 2
        for h in hidden:
            layers += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        self.mlp = nn.Sequential(*layers)
        self.out  = nn.Linear(embed_dim + hidden[-1], 1)
        for emb in [self.gmf_u, self.gmf_i, self.mlp_u, self.mlp_i]:
            nn.init.normal_(emb.weight, std=0.01)

    def forward(self, u, i):
        gmf = self.gmf_u(u) * self.gmf_i(i)
        mlp = self.mlp(torch.cat([self.mlp_u(u), self.mlp_i(i)], dim=1))
        return (torch.sigmoid(self.out(torch.cat([gmf, mlp], dim=1)).squeeze()) * 4.5 + 0.5)

model     = NCFNet(N_USERS, N_MOVIES, EMBED_DIM, HIDDEN, DROPOUT).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)
criterion = nn.MSELoss()
print(f"✅ NCF ready | Params: {sum(p.numel() for p in model.parameters()):,}")

✅ NCF ready | Params: 23,191,105


In [ ]:
# Train

train_losses = []

for epoch in range(1, EPOCHS + 1):
    model.train()
    total = 0
    for u, i, r in tqdm(train_dl, desc=f"Epoch {epoch}/{EPOCHS}"):
        u, i, r = u.to(device), i.to(device), r.to(device)
        optimizer.zero_grad()
        loss = criterion(model(u, i), r)
        loss.backward()
        optimizer.step()
        total += loss.item()
    avg = total / len(train_dl)
    train_losses.append(avg)
    print(f"  Epoch {epoch}: Loss={avg:.4f} | RMSE≈{avg**0.5:.4f}")

print("✅ Training complete")

Epoch 1/10:   0%|          | 0/38642 [00:00<?, ?it/s]

  Epoch 1: Loss=0.6954 | RMSE≈0.8339


Epoch 2/10:   0%|          | 0/38642 [00:00<?, ?it/s]

KeyboardInterrupt: 

: 